# T2.2 Total coherence

**Purpose** (Table 4.2, D-5 validation plan): test whether the total coherence loss estimated by SWB-SCA matches the coherence loss that emerges
from the OceanSAR simulation.

**Implementation**
1. Estimate total coherence loss from SWB-SCA, for the OceanSAR scenario (uniform scene, no thermal noise)
2. Estimate total coherence loss from OceanSAR (ideal raw data)
3. Compare coherences

**Requirements**: reference simulation SWB-SCA, OceanSAR derived total coherence

**Status:** the OceanSAR reference simulation (`dopsca_TSC0d0_U7d30_16x10km`) is not available yet. Steps 2 and 3 skip while `oceansar_dir` is None.

This notebook performs the test and reports the results for interpretation. No tolerances or pass/fail criteria are applied.

In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import scipy.constants
from matplotlib import pyplot as plt

from drama.io import cfg as drcfg
from drama import utils as drtls
import drama.geo as drgeo

from sca_module.dopsca_setup import DopSCAConfig
from sca_module.simulate_L1 import compute_beam_pattern
from leakage.uncertainties import decorrelation

plt.rcParams.update({'font.size': 10, 'font.weight' : 'bold'})

# SWB-SCA coherence
## 0 - Scenario parameters
The SWB-SCA coherence budget is evaluated with the parameters of the OceanSAR scenario, read from its parameter file.

**NOTE** the local parameter file is that of the U = 10 m/s run (`dopsca_TSC0d0_U10d0_16x10km`), while the reference simulation is the U = 7.3 m/s run.
The wind speed is set by hand below. Replace `cfg_file` (and `U`) by the parameter file in the zip once available.

In [2]:
sca = DopSCAConfig()
# the observation cube grid of the other notebooks, which sets the azimuth sampling of the beam
sca.grid_spacing_target = sca.get_grid_spacing_target(magic_divider=2)

# NOTE this parameter file is the U = 10 m/s run, replace by the parameter file of dopsca_TSC0d0_U7d30_16x10km once available
cfg_file = '/Users/opodriscoll/Documents/Scripts/oceansar-1/par/dopsca.cfg'
cfg = drcfg.ConfigFile(cfg_file)

oceansar = dict(
    f0 = cfg.sca.f0,                    # carrier frequency [Hz]
    alt = cfg.sca.alt,                  # orbit altitude [m]
    inc = cfg.sca.inc_angle,            # incidence angle [deg]
    T_pp = cfg.sca.sub_pulse_length,    # sub-pulse separation [s]
    ant_L = cfg.sca.ant_L_tx,           # antenna length [m], ant_L_rx is equal
    U = 7.3,                            # [m/s] from the name dopsca_TSC0d0_U7d30_16x10km, this cfg has wind_U = 10
)
oceansar['Lambda'] = scipy.constants.c / oceansar['f0']

geom = drgeo.QuickRadarGeometry(oceansar['alt'], degrees=True)
oceansar['slant_range'] = geom.inc_to_sr(oceansar['inc'])
# v_ground = auto: OceanSAR moves the platform at the ground-track velocity of a circular orbit
if cfg.sca.v_ground == 'auto':
    v_orbit = np.sqrt(geom.gm_planet / (geom.r_planet + oceansar['alt']))
    oceansar['v'] = v_orbit * geom.r_planet / (geom.r_planet + oceansar['alt'])
else:
    oceansar['v'] = cfg.sca.v_ground

pd.Series(oceansar).to_frame("OceanSAR")

,OceanSAR
f0,5.400000e+09
alt,8.170000e+05
inc,3.500000e+01
T_pp,1.000000e-04
ant_L,3.000000e+00
U,7.300000e+00
Lambda,5.551712e-02
slant_range,9.710183e+05
v,6.597876e+03


## 1 - Azimuth beam and its angular spread
SWB-SCA sets the Doppler decorrelation time from $\sigma_{az}$, the standard deviation of the slant-plane azimuth angle within the -3 dB two-way beam
(as in `add_L1_uncertainties`), sampled on the azimuth grid of the observation cube. OceanSAR's beam is the two-way pattern of a uniformly
illuminated antenna of length `ant_L` (`compute_beam_pattern(..., beam_pattern="sinc")`).

In [3]:
# azimuth grid of the observation cube, as in create_observation_cube
window_size = np.round(sca.az_footprint_cutoff / sca.grid_spacing_target).astype("int")
az_beam = (np.arange(window_size) - (window_size - 1) / 2) * sca.grid_spacing_target
az_angle = xr.DataArray(np.arcsin(az_beam / oceansar['slant_range']), dims=sca.dim_az_beam)

beam = xr.DataArray(
    np.asarray(compute_beam_pattern(
        az_angle_wrt_boresight_slant=az_angle.values,
        grg_angle_wrt_boresight=0,
        antenna_elements=sca.antenna_elements,
        antenna_weighting=sca.antenna_weighting,
        antenna_length=oceansar['ant_L'],
        antenna_height=sca.antenna_height,
        f0=oceansar['f0'],
        beam_pattern="sinc",
    )),
    dims=sca.dim_az_beam,
)

# standard deviation of the azimuth angle within the -3 dB beam, as in add_L1_uncertainties
beam_3dB = xr.where(beam >= 10 ** (-3 / 10) * beam.max(dim=sca.dim_az_beam), 1, np.nan) * az_angle
sigma_az_angle = float(beam_3dB.std(dim=sca.dim_az_beam))

## 2 - Coherence budget
$\gamma = \gamma_{SCR}\,\gamma_{temporal}\,\gamma_D$, with $\gamma_{SCR} = 0.5$ for a uniform scene, $\gamma_{temporal} = \exp\!\big(-(T_{pp}/T_s)^2\big)$, $T_s = 3.29\,\lambda/U$, and
$\gamma_D = \exp\!\big(-(T_{pp}/T_D)^2\big)$, $T_D = 1/(\sqrt{2}\,k\,v\,\sigma_{az})$, as in `add_L1_uncertainties`.

In [4]:
wavenumber = 2 * np.pi / oceansar['Lambda']
T_corr_Doppler = 1 / (np.sqrt(2) * wavenumber * oceansar['v'] * sigma_az_angle)
T_corr_surface = 3.29 * oceansar['Lambda'] / oceansar['U']

gamma_SCR = 0.5
gamma_temporal = decorrelation(oceansar['T_pp'], T_corr_surface)
gamma_D = decorrelation(oceansar['T_pp'], T_corr_Doppler)
gamma_swb_sca = gamma_SCR * gamma_temporal * gamma_D

# OceanSAR coherence
## 3 - Coherence from the OceanSAR raw data
`dpsca_coh` is copied from `oceansar-1/Notebooks/dopsca_view.ipynb`. It correlates each range sample with the sample one sub-pulse length later,
averages over the pulses (azimuth) and smooths over `rg_ml` range samples. With `raw=True` it uses the ideal raw data (`raw_data_r`/`raw_data_i`), without thermal noise.

Set `oceansar_dir` to the unzipped simulation folder to run this section and the comparison.

In [5]:
oceansar_dir = None # e.g. the unzipped dopsca_TSC0d0_U7d30_16x10km folder
raw_file = 'sys_raw_data.nc' # holds the ideal raw data (raw_data_r/i), as read in dopsca_view.ipynb
rg_ml = 200 # range samples smoothed, as in dopsca_view.ipynb
pol_index = 1 # as in dopsca_view.ipynb, NOTE check the polarisation order in the raw file
engine = 'h5netcdf' # as in dopsca_view.ipynb, NOTE needs h5py, which is not installed in the sca_test environment

In [6]:
def dpsca_coh(scaraw, rg_ml=None, raw=False):
    fs = scaraw.rg_sampling.values
    print("Sampling frequency: %f", fs)
    subpulse_length = scaraw.subpulse_length.values
    offset_smp = subpulse_length * fs
    if raw:
        prof =  scaraw['raw_data_r'].values + 1j * scaraw['raw_data_i'].values
    else:
        prof = scaraw['craw'].values
    prof_corr = np.roll(prof,-int(offset_smp), axis=2)
    corr = prof_corr * np.conj(prof)
    nrcs_prof = np.abs(prof)**2
    geom = drgeo.QuickRadarGeometry(scaraw.orbit_alt.values)
    sr0 = scaraw.sr0.values
    sr = sr0 + 3e8 * np.arange(scaraw.rg_dim.size) / (2 * scaraw.rg_sampling.values)
    gr = geom.sr_to_gr(sr)
    range_scaling = (sr0 / sr)**2
    if rg_ml is None:
        coh = np.mean(corr[:,:], axis=1) / np.sqrt(np.mean(np.abs(prof[:,:])**2, axis=1) * np.mean(np.abs(prof_corr[:,:])**2, axis=1))
        nrcs_prof = np.mean(nrcs_prof, axis=1)
    else:
        nrcs_prof = drtls.smooth(np.mean(nrcs_prof, axis=1), rg_ml, axis=1)
        corr = drtls.smooth(np.mean(corr, axis=1), rg_ml, axis=1)
        coh = corr/np.sqrt(nrcs_prof*np.roll(nrcs_prof,-int(offset_smp),axis=1))
    rvel = np.angle(coh)/(2*np.pi*scaraw.subpulse_length.values)*3e8/2/scaraw.f0.values
    return nrcs_prof/range_scaling, coh, rvel

In [7]:
if oceansar_dir is None:
    print("OceanSAR output not available: set oceansar_dir to the unzipped dopsca_TSC0d0_U7d30_16x10km folder")
else:
    scaraw = xr.open_dataset(os.path.join(oceansar_dir, raw_file), engine=engine).load()
    geom_raw = drgeo.QuickRadarGeometry(scaraw.orbit_alt.values)
    sr = scaraw.sr0.values + 3e8 * np.arange(scaraw.rg_dim.size) / (2 * scaraw.rg_sampling.values)
    gr_oceansar = geom_raw.sr_to_gr(sr)
    offset_smp = int(scaraw.subpulse_length.values * scaraw.rg_sampling.values)

    _, coh, _ = dpsca_coh(scaraw, rg_ml=rg_ml, raw=True)
    coh_oceansar = np.abs(coh[pol_index])

    # both overlapping echoes come from within the scene only more than one offset from either end of the profile;
    # the correlation also wraps around (np.roll) at the far end, and the smoothing reaches rg_ml / 2 samples
    interior = slice(offset_smp + rg_ml, -(2 * offset_smp + rg_ml))

OceanSAR output not available: set oceansar_dir to the unzipped dopsca_TSC0d0_U7d30_16x10km folder


# Results
## Summary
SWB-SCA coherence budget for the OceanSAR scenario and, once available, the OceanSAR coherence (mean over the interior of the range profile,
where both overlapping echoes come from within the scene).

In [8]:
summary = {
    "gamma_SCR": gamma_SCR,
    "gamma_temporal": gamma_temporal,
    "gamma_D": gamma_D,
    "SWB-SCA gamma": gamma_swb_sca,
    "OceanSAR gamma": float(np.mean(coh_oceansar[interior])) if oceansar_dir is not None else np.nan,
}
pd.Series(summary).round(4).to_frame("OceanSAR scenario")

,OceanSAR scenario
gamma_SCR,0.5000
gamma_temporal,1.0000
gamma_D,0.8722
SWB-SCA gamma,0.4361
OceanSAR gamma,NaN


In [9]:
if oceansar_dir is not None:
    fig, ax = plt.subplots(figsize = (8, 4))
    ax.plot(gr_oceansar / 1e3, coh_oceansar, color = '#2a78d6', lw = 1.5, label = "OceanSAR")
    ax.axhline(gamma_swb_sca, color = 'k', ls = '--', lw = 1.5, label = "SWB-SCA budget")
    ax.axvspan(gr_oceansar[interior][0] / 1e3, gr_oceansar[interior][-1] / 1e3, color = 'grey', alpha = 0.15, label = "interior")
    ax.set_xlabel("Ground Range [km]")
    ax.set_ylabel("Coherence")
    ax.legend()
    plt.tight_layout()